# Denoising Diffusion Probabilistic Model (DDPM) for 2D Point Generation

This notebook implements an improved DDPM from scratch to generate synthetic 2D points matching the "dino" shape distribution. Improvements include a cosine noise schedule and a deeper MLP with time-embedding injection.

## 1. Imports and Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 2. Load and Preprocess Data

In [3]:
# Load the TSV file
data_path = '/content/Datashape.tsv'
df = pd.read_csv(data_path, sep='\t')

# Filter for "dino" shape only
dino_df = df[df['dataset'] == 'dino']
print(f"Number of dino points: {len(dino_df)}")

# Extract x and y coordinates
x_coords = dino_df['x'].values
y_coords = dino_df['y'].values
data = np.stack([x_coords, y_coords], axis=1).astype(np.float32)

# Store original bounds for later denormalization
x_min, x_max = x_coords.min(), x_coords.max()
y_min, y_max = y_coords.min(), y_coords.max()
print(f"X range: [{x_min:.2f}, {x_max:.2f}]")
print(f"Y range: [{y_min:.2f}, {y_max:.2f}]")

# Normalize to [-1, 1]
data_normalized = np.zeros_like(data)
data_normalized[:, 0] = 2 * (data[:, 0] - x_min) / (x_max - x_min) - 1
data_normalized[:, 1] = 2 * (data[:, 1] - y_min) / (y_max - y_min) - 1

# Create PyTorch dataset and dataloader
tensor_data = torch.tensor(data_normalized, dtype=torch.float32)
dataset = TensorDataset(tensor_data)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Data shape: {tensor_data.shape}")
print(f"Normalized data range: [{data_normalized.min():.2f}, {data_normalized.max():.2f}]")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Datashape.tsv'

In [ ]:
# Visualize the original dino data
plt.figure(figsize=(8, 8))
plt.scatter(x_coords, y_coords, s=20, alpha=0.7, c='blue', label='Dino points')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Original Dino Shape')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

## 3. DDPM Implementation

### 3.1 Noise Schedule

We transition to a **Cosine Beta Schedule** which is often better at preserving information in later timesteps compared to a linear schedule.

In [ ]:
class DDPMScheduler:
    """DDPM noise scheduler with Cosine beta schedule."""
    
    def __init__(self, num_timesteps=1000, schedule='cosine', device='cpu'):
        self.num_timesteps = num_timesteps
        self.device = device
        
        if schedule == 'linear':
            self.betas = torch.linspace(1e-4, 0.02, num_timesteps, device=device)
        elif schedule == 'cosine':
            # Cosine schedule from Improved DDPM paper (Nichol & Dhariwal)
            steps = num_timesteps + 1
            x = torch.linspace(0, num_timesteps, steps, device=device)
            alphas_cumprod = torch.cos(((x / num_timesteps) + 0.008) / (1 + 0.008) * np.pi / 2) ** 2
            alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
            betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
            self.betas = torch.clip(betas, 0, 0.999)
        
        # Alpha values
        self.alphas = 1.0 - self.betas
        
        # Cumulative product of alphas (alpha_bar)
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)
        
        # For sampling
        self.alpha_bars_prev = F.pad(self.alpha_bars[:-1], (1, 0), value=1.0)
        
        # Precompute values for forward diffusion
        self.sqrt_alpha_bars = torch.sqrt(self.alpha_bars)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - self.alpha_bars)
        
        # Precompute values for reverse diffusion
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.posterior_variance = self.betas * (1.0 - self.alpha_bars_prev) / (1.0 - self.alpha_bars)
    
    def add_noise(self, x_0, t, noise=None):
        """Forward diffusion: q(x_t | x_0)"""
        if noise is None:
            noise = torch.randn_like(x_0)
        
        sqrt_alpha_bar_t = self.sqrt_alpha_bars[t].view(-1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bars[t].view(-1, 1)
        
        x_t = sqrt_alpha_bar_t * x_0 + sqrt_one_minus_alpha_bar_t * noise
        return x_t, noise
    
    def sample_timesteps(self, batch_size):
        return torch.randint(0, self.num_timesteps, (batch_size,), device=self.device)

# Initialize scheduler with Cosine schedule
num_timesteps = 1000
scheduler = DDPMScheduler(num_timesteps=num_timesteps, schedule='cosine', device=device)
print(f"Scheduler initialized with Cosine schedule ({num_timesteps} steps)")

### 3.2 Denoising Network (Score Model)

We implement a deeper MLP where the time embedding is injected into every layer, not just the input. This helps the network adapt its denoising strategy more effectively for different noise levels.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, time_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.SiLU(),
            nn.Linear(dim, dim)
        )
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, dim)
        )
    
    def forward(self, x, t_emb):
        # Inject time embedding
        h = self.mlp(x) + self.time_mlp(t_emb)
        return x + h


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    
    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        embeddings = np.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = t[:, None] * embeddings[None, :]
        embeddings = torch.cat([torch.sin(embeddings), torch.cos(embeddings)], dim=-1)
        return embeddings


class DenoisingMLP(nn.Module):
    """Deep MLP with Time Embedding Injection into every block."""
    
    def __init__(self, input_dim=2, hidden_dim=512, time_emb_dim=256, num_blocks=6):
        super().__init__()
        
        # Time embedding MLP
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )
        
        # Initial projection
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        
        # Residual blocks with time injection
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, time_emb_dim) for _ in range(num_blocks)
        ])
        
        # Output layer
        self.output_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def forward(self, x, t):
        # Compute time embedding once
        t_emb = self.time_mlp(t)
        
        # Initial projection of 2D coordinates
        h = self.input_layer(x)
        
        # Pass through residual blocks with time injection
        for block in self.blocks:
            h = block(h, t_emb)
        
        # Output
        return self.output_layer(h)


# Initialize improved model
model = DenoisingMLP(input_dim=2, hidden_dim=512, time_emb_dim=256, num_blocks=6).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

### 3.3 Reverse Diffusion (Sampling)

Standard DDPM sampling algorithm.

In [ ]:
@torch.no_grad()
def sample(model, scheduler, num_samples, sample_dim=2):
    model.eval()
    x = torch.randn(num_samples, sample_dim, device=scheduler.device)
    
    for t in tqdm(reversed(range(scheduler.num_timesteps)), total=scheduler.num_timesteps, desc="Sampling"):
        t_batch = torch.full((num_samples,), t, device=scheduler.device, dtype=torch.long)
        
        # Predict noise
        predicted_noise = model(x, t_batch)
        
        # Get scheduler values
        alpha_t = scheduler.alphas[t]
        alpha_bar_t = scheduler.alpha_bars[t]
        beta_t = scheduler.betas[t]
        
        # mu = (1 / sqrt(alpha_t)) * (x_t - (beta_t / sqrt(1 - alpha_bar_t)) * eps_theta)
        mean = (1.0 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * predicted_noise)
        
        if t > 0:
            noise = torch.randn_like(x)
            sigma = torch.sqrt(scheduler.posterior_variance[t])
            x = mean + sigma * noise
        else:
            x = mean
    
    return x

print("Sampling function defined.")

## 4. Training Loop

In [ ]:
def train(model, dataloader, scheduler, num_epochs=10000, lr=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler_lr = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    losses = []
    model.train()
    
    for epoch in tqdm(range(num_epochs), desc="Training"):
        epoch_loss = 0.0
        for batch in dataloader:
            x_0 = batch[0].to(device)
            
            # Data augmentation: add very small noise to x_0 to smooth the manifold
            x_0 = x_0 + torch.randn_like(x_0) * 0.005
            
            t = scheduler.sample_timesteps(x_0.shape[0])
            noise = torch.randn_like(x_0)
            x_t, _ = scheduler.add_noise(x_0, t, noise)
            
            predicted_noise = model(x_t, t)
            loss = F.mse_loss(predicted_noise, noise)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        scheduler_lr.step()
        losses.append(epoch_loss / len(dataloader))
        
        if (epoch + 1) % 1000 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {losses[-1]:.6f}")
    
    return losses

print("Training function defined.")

In [ ]:
# Train for more epochs with the improved architecture
num_epochs = 12000
print(f"Starting training for {num_epochs} epochs...")
losses = train(model, dataloader, scheduler, num_epochs=num_epochs, lr=2e-4)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training Loss over Epochs')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

## 5. Generate Samples and Visualization

In [ ]:
num_samples = 1000
generated_samples = sample(model, scheduler, num_samples=num_samples).cpu().numpy()

# Denormalize
generated_denorm = np.zeros_like(generated_samples)
generated_denorm[:, 0] = (generated_samples[:, 0] + 1) / 2 * (x_max - x_min) + x_min
generated_denorm[:, 1] = (generated_samples[:, 1] + 1) / 2 * (y_max - y_min) + y_min

In [ ]:
plt.figure(figsize=(12, 10))
plt.scatter(x_coords, y_coords, s=50, alpha=0.8, c='blue', label='Real Dino Points', edgecolors='white', linewidths=0.5)
plt.scatter(generated_denorm[:, 0], generated_denorm[:, 1], s=20, alpha=0.4, c='red', label='Generated Points')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.title('Improved DDPM: Cosine Schedule + Layer Injection')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(x_coords, y_coords, s=30, alpha=0.8, c='blue', edgecolors='white', linewidths=0.5)
axes[0].set_title('Real Dino Points')
axes[0].axis('equal')
axes[1].scatter(generated_denorm[:, 0], generated_denorm[:, 1], s=10, alpha=0.6, c='red')
axes[1].set_title('Generated Points')
axes[1].axis('equal')
plt.show()